# Labelling defect candidates and training a classifier

This notebook detects candidate defects, lets you label them as
**real / uncertain / artefact**, re-extracts their features from the real
director field, performs leave-one-image-out validation, and saves a
reusable classifier.

## Why one physical defect can look like several marks

The winding test is evaluated on a spatial grid and at several smoothing
scales. A large core can therefore trigger several nearby detections. The
detector already clusters detections of the **same charge** within
`defect_cluster_radius_px` and replaces them with one centroid. Each marker
shown here is normally one such multiscale cluster, not one raw grid hit.

Occasionally a broad core is split into two final markers. The detection
cell reports same-charge markers closer than `duplicate_radius_px` and can
merge them before labelling. Merging is deliberately off by default:
nearby defects can be genuinely distinct, and a nearby `+1/2`/`-1/2` pair
is expected physics rather than a duplicate. Opposite charges are never
merged.

Use persistent Google Drive storage if the labelling will span multiple
sessions. Run all cells in order the first time.

In [ ]:
#@title 1 · Setup repository and dependencies { display-mode: "form" }
import os
import subprocess
import sys

REPO = "https://github.com/Danpc11/lung-nematic.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ROOT = "/content/lung-nematic"

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(
        ["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", ROOT, "reset", "--hard", f"origin/{BRANCH}"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO, ROOT],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ROOT, "ipympl"],
    check=True,
)

for name in [
    module for module in list(sys.modules)
    if module == "lung_nematic" or module.startswith("lung_nematic.")
]:
    del sys.modules[name]
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
from lung_nematic.config import load_default_config
from lung_nematic.defect_classifier import (
    CLASSES,
    grouped_cross_validate,
    train_classifier,
)
from lung_nematic.defect_features import extract_features
from lung_nematic.labeling import LabelingSession, load_labels

commit = subprocess.run(
    ["git", "-C", ROOT, "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Ready at commit {commit}. Class vocabulary: {CLASSES}")

In [ ]:
#@title 2 · Persistent workspace and images { display-mode: "form" }
from pathlib import Path

storage = "Google Drive (persistent)"  #@param ["Google Drive (persistent)", "Colab session (temporary)"]
drive_workspace = "MyDrive/lung_nematic_labelling"  #@param {type:"string"}
image_source = "Use images already in workspace"  #@param ["Use images already in workspace", "Upload images now"]

if storage.startswith("Google Drive"):
    from google.colab import drive

    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive") / drive_workspace
else:
    WORKSPACE = Path("/content/lung_nematic_labelling")

IMAGES_DIR = WORKSPACE / "images"
LABELS_DIR = WORKSPACE / "labels"
MODELS_DIR = WORKSPACE / "models"
for directory in (IMAGES_DIR, LABELS_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if image_source == "Upload images now":
    from google.colab import files

    uploaded = files.upload()
    for filename, contents in uploaded.items():
        destination = IMAGES_DIR / Path(filename).name
        destination.write_bytes(contents)
        print(f"saved {destination}")

EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
image_files = sorted(
    path for path in IMAGES_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in EXTENSIONS
)
print(f"Workspace: {WORKSPACE}")
print(f"{len(image_files)} image(s) available:")
for path in image_files:
    print("  ", path.relative_to(IMAGES_DIR))
if not image_files:
    print("Upload images now or place them in the workspace/images folder.")

In [ ]:
#@title 3 · Detect, inspect duplicates, and open the labeller { display-mode: "form" }
%matplotlib widget
from dataclasses import replace
from PIL import Image
from sklearn.cluster import DBSCAN

Image.MAX_IMAGE_PIXELS = None

image_file = ""  #@param {type:"string"}
image_id_override = ""  #@param {type:"string"}
image_kind = "histology (collagen)"  #@param ["histology (collagen)", "phase-contrast gel"]

sigmas_px = "18, 25, 32"  #@param {type:"string"}
density_quantile = 0.30  #@param {type:"slider", min:0, max:0.9, step:0.05}
min_scales_for_persistence = 2  #@param {type:"slider", min:1, max:4, step:1}
defect_grid_step_px = 12  #@param {type:"slider", min:4, max:64, step:4}
min_edge_distance_px = 25  #@param {type:"slider", min:0, max:120, step:5}
defect_cluster_radius_px = 70.0  #@param {type:"slider", min:10, max:200, step:5}

#@markdown Optional second pass over already-clustered final markers.
duplicate_radius_px = 70.0  #@param {type:"slider", min:10, max:250, step:5}
merge_possible_duplicates = False  #@param {type:"boolean"}

if not image_files:
    raise FileNotFoundError("No images are available. Run cell 2 first.")
if image_file:
    requested = Path(image_file)
    if requested.is_absolute() and requested.exists():
        image_path = requested
    else:
        matches = [
            path for path in image_files
            if str(path.relative_to(IMAGES_DIR)) == image_file
            or path.name == image_file
        ]
        if len(matches) != 1:
            raise ValueError(
                f"image_file matched {len(matches)} files; use its path relative "
                "to workspace/images to select exactly one."
            )
        image_path = matches[0]
else:
    image_path = image_files[0]
    print(f"image_file was blank; using {image_path.relative_to(IMAGES_DIR)}")

parsed_sigmas = tuple(float(value.strip()) for value in sigmas_px.split(",") if value.strip())
cfg = replace(
    load_default_config(),
    sigmas_px=parsed_sigmas,
    density_quantile=float(density_quantile),
    min_scales_for_persistence=min(
        int(min_scales_for_persistence), len(parsed_sigmas)
    ),
    defect_grid_step_px=int(defect_grid_step_px),
    min_edge_distance_px=int(min_edge_distance_px),
    defect_cluster_radius_px=float(defect_cluster_radius_px),
)
cfg.validate()

rgb = np.asarray(Image.open(image_path).convert("RGB"))
if image_kind.startswith("phase"):
    from lung_nematic.phase_contrast import analyze_phase_contrast

    analysis = analyze_phase_contrast(rgb, cfg)
    field = analysis["field"]
    candidates = analysis["defects"].copy()
else:
    from lung_nematic.collagen_field import detect_multiscale_collagen_defects
    from lung_nematic.preprocessing import make_tissue_mask

    tissue_mask, hed = make_tissue_mask(rgb)
    candidates, fields, _ = detect_multiscale_collagen_defects(
        hed[:, :, 1], tissue_mask, cfg
    )
    representative_sigma = float(parsed_sigmas[len(parsed_sigmas) // 2])
    field = fields[representative_sigma]

if candidates.empty:
    raise RuntimeError(
        "No persistent candidates were detected. Inspect the image and "
        "consider lowering min_scales_for_persistence or adjusting scales."
    )

def assign_same_charge_groups(frame, radius):
    grouped = frame.copy()
    grouped["possible_duplicate_group"] = -1
    next_group = 0
    for charge in sorted(grouped["charge"].unique()):
        indices = grouped.index[grouped["charge"] == charge]
        points = grouped.loc[indices, ["x_px", "y_px"]].to_numpy(float)
        labels = DBSCAN(eps=float(radius), min_samples=1).fit(points).labels_
        for local_group in sorted(set(labels)):
            members = indices[labels == local_group]
            grouped.loc[members, "possible_duplicate_group"] = next_group
            next_group += 1
    return grouped

candidates = assign_same_charge_groups(candidates, duplicate_radius_px)
group_sizes = candidates.groupby("possible_duplicate_group").size()
duplicate_ids = group_sizes[group_sizes > 1].index
possible_duplicates = candidates.loc[
    candidates["possible_duplicate_group"].isin(duplicate_ids),
    ["possible_duplicate_group", "defect_id", "x_px", "y_px", "charge"],
]

if possible_duplicates.empty:
    print("No possible same-charge duplicate markers at this radius.")
else:
    print("Possible duplicates (inspect before enabling merge):")
    display(possible_duplicates)

def merge_same_charge_groups(frame):
    records = []
    for _, members in frame.groupby("possible_duplicate_group", sort=True):
        record = members.iloc[0].copy()
        record["x_px"] = float(members["x_px"].mean())
        record["y_px"] = float(members["y_px"].mean())
        for column in ("mean_local_order", "mean_edge_distance_px"):
            if column in members:
                record[column] = float(members[column].mean())
        for column in ("scales_detected", "scale_fraction", "confidence"):
            if column in members:
                record[column] = members[column].max()
        if "sigmas_px" in members:
            values = set()
            for item in members["sigmas_px"].dropna().astype(str):
                values.update(part.strip() for part in item.split(",") if part.strip())
            record["sigmas_px"] = ",".join(
                sorted(values, key=lambda value: float(value))
            )
        record["merged_marker_count"] = int(len(members))
        records.append(record)
    merged = pd.DataFrame(records).reset_index(drop=True)
    merged["defect_id"] = np.arange(1, len(merged) + 1)
    return merged

before_merge = len(candidates)
if merge_possible_duplicates:
    candidates = merge_same_charge_groups(candidates)
    print(f"Merged {before_merge} final markers into {len(candidates)} candidates.")
else:
    candidates["merged_marker_count"] = 1
    print(
        f"Keeping all {len(candidates)} final markers. Enable merging only "
        "after visually confirming the groups above."
    )

image_id = image_id_override.strip() or image_path.stem
candidates["source_path"] = str(image_path)
candidates["image_kind"] = image_kind
candidates["representative_sigma_px"] = float(parsed_sigmas[len(parsed_sigmas) // 2])
candidates["collagen_inner_scale_px"] = float(cfg.collagen_inner_scale_px)
candidates["mask_normalized_smoothing"] = bool(cfg.mask_normalized_smoothing)

session = LabelingSession(rgb, field, candidates, image_id=image_id)
label_path = LABELS_DIR / f"{image_id}.csv"
if label_path.exists():
    existing = pd.read_csv(label_path)
    if {"candidate_id", "label"}.issubset(existing.columns):
        saved_labels = existing.set_index("candidate_id")["label"].to_dict()
        session.labels = [saved_labels.get(candidate_id) for candidate_id in session._id_of]
        restored = sum(label is not None for label in session.labels)
        print(f"Restored {restored} existing labels from {label_path.name}.")

print(f"{len(candidates)} candidates in {image_id}")
session.show()

In [ ]:
#@title 4 · Save labels for this image { display-mode: "form" }
download_labels_now = False  #@param {type:"boolean"}

saved = session.save(label_path)
if saved.empty:
    print("No labelled candidates were saved; click markers before saving.")
elif download_labels_now:
    from google.colab import files

    files.download(str(label_path))
print(f"Persistent location: {label_path}")

## Label additional images

Change `image_file` in cell 3 and repeat cells 3–4. Use a unique
`image_id_override` when different folders contain images with the same
filename. Aim for multiple images and examples of both real defects and
artefacts; validation cannot estimate generalisation from one image.

In [ ]:
#@title 5 · Re-extract features from every real image { display-mode: "form" }
from dataclasses import replace
from PIL import Image

feature_core_radius_px = 25.0  #@param {type:"slider", min:5, max:100, step:5}

label_paths = sorted(LABELS_DIR.glob("*.csv"))
if not label_paths:
    raise FileNotFoundError(f"No label CSV files found in {LABELS_DIR}.")
labelled = load_labels(label_paths)
if labelled.empty:
    raise ValueError("The label files contain no labelled candidates.")

available_images = sorted(
    path for path in IMAGES_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in EXTENSIONS
)

def resolve_source(group):
    if "source_path" in group:
        saved_paths = group["source_path"].dropna().astype(str).unique()
        if len(saved_paths) == 1 and Path(saved_paths[0]).exists():
            return Path(saved_paths[0])
    image_id = str(group["image_id"].iloc[0])
    matches = [path for path in available_images if path.stem == image_id]
    if len(matches) != 1:
        raise ValueError(
            f"Cannot uniquely resolve the image for {image_id}: {len(matches)} matches. "
            "Re-label it with this notebook so source_path is saved."
        )
    return matches[0]

def representative_field(group):
    source = resolve_source(group)
    rgb = np.asarray(Image.open(source).convert("RGB"))
    representative_sigma = float(
        group.get("representative_sigma_px", pd.Series([25.0])).iloc[0]
    )
    kind = str(group.get("image_kind", pd.Series(["histology (collagen)"])).iloc[0])
    local_cfg = replace(
        load_default_config(),
        sigmas_px=(representative_sigma,),
        min_scales_for_persistence=1,
    )
    if kind.startswith("phase"):
        from lung_nematic.phase_contrast import phase_contrast_field, cell_texture_mask

        coverage = cell_texture_mask(rgb)
        return phase_contrast_field(rgb, representative_sigma, mask=coverage)

    from lung_nematic.collagen_field import compute_collagen_field
    from lung_nematic.preprocessing import make_tissue_mask

    tissue_mask, hed = make_tissue_mask(rgb)
    inner_scale = float(
        group.get(
            "collagen_inner_scale_px",
            pd.Series([local_cfg.collagen_inner_scale_px]),
        ).iloc[0]
    )
    mask_normalized = bool(
        group.get("mask_normalized_smoothing", pd.Series([False])).iloc[0]
    )
    return compute_collagen_field(
        hed[:, :, 1],
        representative_sigma,
        inner_scale,
        tissue_mask=tissue_mask,
        mask_normalized=mask_normalized,
    )

feature_rows = []
for image_id, group in labelled.groupby("image_id", sort=True):
    field = representative_field(group)
    features = extract_features(
        group.reset_index(drop=True),
        field,
        core_radius_px=float(feature_core_radius_px),
    )
    features["image_id"] = image_id
    features["label"] = group["label"].to_numpy()
    feature_rows.append(features)
    print(f"{image_id}: {len(features)} labelled candidates")

data = pd.concat(feature_rows, ignore_index=True)
print(f"Feature matrix: {data.shape}")
print("Label mix:", data["label"].value_counts().to_dict())
print("Images:", data["image_id"].nunique())
display(data.head())

In [ ]:
#@title 6 · Validate, train, and save classifier { display-mode: "form" }
import json

classifier_kind = "random_forest"  #@param ["random_forest", "logistic"]
uncertain_policy = "exclude from training"  #@param ["exclude from training", "train as third class"]
classifier_seed = 0  #@param {type:"integer"}

if uncertain_policy.startswith("exclude"):
    training_data = data.loc[data["label"] != "uncertain"].copy()
else:
    training_data = data.copy()

if training_data["label"].nunique() < 2:
    raise ValueError(
        "Training needs at least two classes. Label both real defects and artefacts."
    )
if training_data["image_id"].nunique() < 2:
    raise ValueError(
        "Leave-one-image-out validation needs labels from at least two images."
    )

validation = grouped_cross_validate(
    training_data,
    training_data["label"],
    training_data["image_id"],
    kind=classifier_kind,
    seed=int(classifier_seed),
)
confusion = pd.DataFrame(
    validation["confusion_matrix"],
    index=[f"true_{name}" for name in validation["confusion_labels"]],
    columns=[f"pred_{name}" for name in validation["confusion_labels"]],
)
print("Leave-one-image-out confusion matrix:")
display(confusion)
print("Per-image validation:")
display(validation["per_image"])

classifier = train_classifier(
    training_data,
    training_data["label"],
    kind=classifier_kind,
    seed=int(classifier_seed),
)
importance = classifier.feature_importance()
print("Feature importance:")
display(importance)

uncertain = data.loc[data["label"] == "uncertain"]
if not uncertain.empty:
    uncertain_scores = classifier.predict(uncertain)
    print("Held-out uncertain examples:")
    display(
        uncertain_scores[
            ["image_id", "candidate_id", "p_artefact", "p_uncertain", "p_real"]
        ]
    )

model_stem = MODELS_DIR / "defect_classifier"
classifier.metadata.update(
    {
        "uncertain_policy": uncertain_policy,
        "n_images": int(training_data["image_id"].nunique()),
        "seed": int(classifier_seed),
    }
)
classifier.save(model_stem)
data.to_csv(MODELS_DIR / "training_features.csv", index=False)
importance.to_csv(MODELS_DIR / "feature_importance.csv", index=False)
validation["per_image"].to_csv(
    MODELS_DIR / "validation_per_image.csv", index=False
)
with (MODELS_DIR / "validation_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "report": validation["report"],
            "confusion_labels": validation["confusion_labels"],
            "confusion_matrix": validation["confusion_matrix"].tolist(),
            "n_images": validation["n_images"],
            "n_candidates": validation["n_candidates"],
            "uncertain_policy": uncertain_policy,
        },
        handle,
        indent=2,
    )
print(f"Saved classifier and reports under {MODELS_DIR}")

In [ ]:
#@title 7 · Download model, validation, and labels { display-mode: "form" }
import zipfile

bundle = Path("/content/defect_labelling_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(MODELS_DIR.glob("*")):
        if path.is_file():
            archive.write(path, f"models/{path.name}")
    for path in sorted(LABELS_DIR.glob("*.csv")):
        archive.write(path, f"labels/{path.name}")

print(f"Bundle ready: {bundle} ({bundle.stat().st_size / 1e6:.2f} MB)")
from google.colab import files

files.download(str(bundle))

## Interpreting the result

The leave-one-image-out confusion matrix is the generalisation check;
in-sample accuracy is not informative here. Feature importance is useful
only after validation shows that the model transfers to unseen images.

If uncertain examples were excluded, their probabilities show where they
fall relative to the clean real/artefact boundary. If they were trained as
a third class, ensure that every validation fold contains enough examples
to learn that class.

A merged marker represents one user-approved physical core and records
`merged_marker_count`. Keep merging conservative and document the radius,
because automatic single-link grouping can combine two genuinely distinct
same-charge defects when they are very close.